# Equity Data Validation Notebook

This notebook validates the 5 equity CSV files in `data/` for schema, date integrity, missing values, and summary statistics.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
DATA_DIR = PROJECT_ROOT / "data"

TICKERS = ["AAPL", "AMZN", "GOOGL", "MSFT", "TSLA"]
REQUIRED_COLUMNS = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
ADJ_CLOSE_ALIASES = {"Adj_Close": "Adj Close", "AdjClose": "Adj Close"}

MISSING_FAIL_COLUMNS = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {DATA_DIR}")

Project root: C:\Users\Hyperion\GBM Project\PortfolioModeling-GBM
Data dir: C:\Users\Hyperion\GBM Project\PortfolioModeling-GBM\data


In [2]:
def load_and_normalize_csv(ticker: str, data_dir: Path) -> pd.DataFrame:
    path = data_dir / f"{ticker}_daily_5y.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    df = pd.read_csv(path)
    df = df.rename(columns={k: v for k, v in ADJ_CLOSE_ALIASES.items() if k in df.columns})

    if "Date" not in df.columns:
        raise ValueError(f"{ticker}: missing 'Date' column")

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    bad_dates = int(df["Date"].isna().sum())
    if bad_dates > 0:
        raise ValueError(f"{ticker}: found {bad_dates} unparseable dates")

    for col in ["Open", "High", "Low", "Close", "Adj Close", "Volume"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def validate_ticker_df(ticker: str, df: pd.DataFrame) -> tuple[dict, pd.DataFrame, pd.DataFrame]:
    result = {
        "ticker": ticker,
        "file_present": True,
        "schema_ok": True,
        "date_sorted_ok": True,
        "date_duplicate_count": 0,
        "missing_ok": True,
        "status": "PASS",
        "notes": "",
    }

    missing_cols = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing_cols:
        result["schema_ok"] = False
        result["status"] = "FAIL"
        result["notes"] += f"Missing required columns: {missing_cols}. "

    is_sorted = df["Date"].is_monotonic_increasing
    if not is_sorted:
        result["date_sorted_ok"] = False
        if result["status"] != "FAIL":
            result["status"] = "PASS WITH WARNINGS"
        result["notes"] += "Dates not sorted ascending. "
        df = df.sort_values("Date").reset_index(drop=True)

    dup_count = int(df.duplicated(subset=["Date"]).sum())
    result["date_duplicate_count"] = dup_count
    if dup_count > 0:
        result["status"] = "FAIL"
        result["notes"] += f"Duplicate dates found: {dup_count}. "

    missing_summary = df[MISSING_FAIL_COLUMNS].isna().sum().to_dict() if all(c in df.columns for c in MISSING_FAIL_COLUMNS) else {}
    missing_total = int(sum(missing_summary.values())) if missing_summary else 0
    if missing_total > 0:
        result["missing_ok"] = False
        result["status"] = "FAIL"
        result["notes"] += f"Missing values in required columns: {missing_summary}. "

    date_range_summary = {
        "ticker": ticker,
        "rows": len(df),
        "start_date": df["Date"].min().date().isoformat() if len(df) else None,
        "end_date": df["Date"].max().date().isoformat() if len(df) else None,
    }

    stats_cols = [c for c in ["Open", "High", "Low", "Close", "Adj Close", "Volume"] if c in df.columns]
    summary_stats = df[stats_cols].describe().T.reset_index().rename(columns={"index": "column"}) if stats_cols else pd.DataFrame()
    summary_stats.insert(0, "ticker", ticker)

    return result, pd.DataFrame([date_range_summary]), summary_stats


In [3]:
validation_rows = []
date_ranges = []
all_summary_stats = []

for ticker in TICKERS:
    try:
        df = load_and_normalize_csv(ticker, DATA_DIR)
        row, dr, stats = validate_ticker_df(ticker, df)
    except Exception as exc:
        row = {
            "ticker": ticker,
            "file_present": False,
            "schema_ok": False,
            "date_sorted_ok": False,
            "date_duplicate_count": None,
            "missing_ok": False,
            "status": "FAIL",
            "notes": str(exc),
        }
        dr = pd.DataFrame([{"ticker": ticker, "rows": 0, "start_date": None, "end_date": None}])
        stats = pd.DataFrame()

    validation_rows.append(row)
    date_ranges.append(dr)
    if not stats.empty:
        all_summary_stats.append(stats)

validation_df = pd.DataFrame(validation_rows)
date_ranges_df = pd.concat(date_ranges, ignore_index=True)
summary_stats_df = pd.concat(all_summary_stats, ignore_index=True) if all_summary_stats else pd.DataFrame()

status_priority = {"FAIL": 3, "PASS WITH WARNINGS": 2, "PASS": 1}
overall_status = "PASS"
if (validation_df["status"] == "FAIL").any():
    overall_status = "FAIL"
elif (validation_df["status"] == "PASS WITH WARNINGS").any():
    overall_status = "PASS WITH WARNINGS"

print("=== VALIDATION STATUS ===")
print(validation_df[["ticker", "status", "schema_ok", "date_sorted_ok", "date_duplicate_count", "missing_ok", "notes"]].to_string(index=False))
print("\n=== DATE RANGE SUMMARY ===")
print(date_ranges_df.to_string(index=False))

print("\n=== SUMMARY STATISTICS (all tickers) ===")
if summary_stats_df.empty:
    print("No summary statistics available (all loads failed).")
else:
    print(summary_stats_df[["ticker", "column", "count", "mean", "std", "min", "25%", "50%", "75%", "max"]].to_string(index=False))

coverage_rows = date_ranges_df[date_ranges_df["rows"] > 0]
if not coverage_rows.empty:
    overall_start = coverage_rows["start_date"].min()
    overall_end = coverage_rows["end_date"].max()
    print("\n=== CROSS-TICKER COVERAGE ===")
    print(f"Earliest start date: {overall_start}")
    print(f"Latest end date:   {overall_end}")

print(f"\n=== OVERALL STATUS: {overall_status} ===")


=== VALIDATION STATUS ===


ticker status  schema_ok  date_sorted_ok  date_duplicate_count  missing_ok notes
  AAPL   PASS       True            True                     0        True      
  AMZN   PASS       True            True                     0        True      
 GOOGL   PASS       True            True                     0        True      
  MSFT   PASS       True            True                     0        True      
  TSLA   PASS       True            True                     0        True      

=== DATE RANGE SUMMARY ===
ticker  rows start_date   end_date
  AAPL  1304 2021-04-23 2026-04-22
  AMZN  1304 2021-04-23 2026-04-22
 GOOGL  1304 2021-04-23 2026-04-22
  MSFT  1304 2021-04-23 2026-04-22
  TSLA  1304 2021-04-23 2026-04-22

=== SUMMARY STATISTICS (all tickers) ===
ticker    column  count         mean          std          min          25%          50%          75%          max
  AAPL      Open 1304.0 1.894859e+02 4.121139e+01 1.231600e+02 1.537750e+02 1.800200e+02 2.240025e+02 2.862000e+02
  AA